In [1]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

petit_modele = ChatOllama(model="qwen3:1.7b", temperature=0, reasoning=True)
grand_modele = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

def derniere_reponse(resultat):
    return resultat["messages"][-1].content

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest

@wrap_model_call
def choisir_modele(request: ModelRequest, handler):
    question = request.messages[-1].content
    complexe = len(question) > 80 or any(m in question.lower() for m in ["explique", "compare", "analyse"])
    modele = grand_modele if complexe else petit_modele
    print(f"  [dynamic_model] -> {modele.model}")
    return handler(request.override(model=modele))   # on remplace le modèle pour CET appel

agent = create_agent(model=petit_modele, middleware=[choisir_modele])

print(derniere_reponse(agent.invoke({"messages": [HumanMessage("Bonjour !")]})))
print(derniere_reponse(agent.invoke({"messages": [HumanMessage("Explique la différence entre un RNN et un Transformer.")]})))

  [dynamic_model] -> qwen3:1.7b
Bonjour à toi ! 😊
  [dynamic_model] -> qwen3:4b
### Différence entre un RNN et un Transformer

#### 1. **Mechanisme de traitement**
- **RNN (Réseau de Neurones Récurrents)** :  
  Traite les séquences **de manière séquentielle** (un pas à la fois). Chaque étape dépend du résultat de l'étape précédente via un *état caché* (hidden state).  
  *Exemple* : Pour un texte, le mot "chat" influence le mot "joue" dans l'étape suivante.

- **Transformer** :  
  Traite **tous les tokens en parallèle** (pas de dépendance séquentielle). Utilise des mécanismes d'**attention** pour permettre à chaque token de s'exprimer en fonction de l'ensemble des autres tokens.  
  *Exemple* : Dans un texte, le mot "chat" peut influencer directement le mot "chien" sans passer par les mots intermédiaires.

---

#### 2. **Gestion des dépendances**
- **RNN** :  
  Peut avoir des **problèmes de dépendances à long terme** (vanishing gradients). Les informations des mots très éloignés dan

In [3]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt

@dataclass
class Contexte:
    niveau: str = "debutant"

@dynamic_prompt
def prompt_selon_niveau(request: ModelRequest) -> str:
    niveau = request.runtime.context.niveau
    print(f"  [dynamic_prompt] niveau = {niveau}")
    if niveau == "expert":
        return "Tu es un assistant technique. Réponds avec le vocabulaire précis, en 2 phrases, en français."
    return "Tu es un professeur patient. Explique avec une image simple de la vie courante, en 2 phrases, en français."

agent = create_agent(model=grand_modele, middleware=[prompt_selon_niveau], context_schema=Contexte)

question = {"messages": [HumanMessage("C'est quoi l'attention dans un Transformer ?")]}
print(derniere_reponse(agent.invoke(question, context=Contexte(niveau="debutant"))))
print()
print(derniere_reponse(agent.invoke(question, context=Contexte(niveau="expert"))))

  [dynamic_prompt] niveau = debutant
L'attention dans un Transformer est comme chercher un jouet spécifique dans une boîte de jouets : le modèle se concentre sur les parties les plus pertinentes de l'entrée pour comprendre le contexte. Cela permet de relier les informations importantes, comme quand on se souvient d'un détail clé dans un livre.

  [dynamic_prompt] niveau = expert
L'attention dans un Transformer est un mécanisme permettant aux tokens d'une séquence d'identifier les éléments les plus pertinents pour le traitement contextuel. Elle calcule des poids d'attention via des matrices query, key et value afin de déterminer l'importance relative de chaque token dans la séquence.


In [4]:
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call

@tool
def diviser(a: float, b: float) -> float:
    """Divise a par b."""
    return a / b

@wrap_tool_call
def gerer_erreurs_outils(request, handler):
    try:
        return handler(request)                      # on exécute l'outil normalement
    except Exception as e:
        print(f"  [tool_error] {request.tool_call['name']} a échoué : {e}")
        return ToolMessage(                          # au lieu de planter, on prévient le LLM
            content=f"Erreur dans l'outil : {e}. Explique le problème à l'utilisateur.",
            tool_call_id=request.tool_call["id"],
        )

agent = create_agent(
    model=grand_modele,
    tools=[diviser],
    middleware=[gerer_erreurs_outils],
    system_prompt="Utilise l'outil diviser pour les divisions. Réponds en français.",
)
print(derniere_reponse(agent.invoke({"messages": [HumanMessage("Combien font 10 divisé par 0 ?")]})))

La division par zéro est impossible. 10 divisé par 0 n'est pas défini.


In [5]:
from langchain.agents.middleware import before_agent, PIIMiddleware

MOTS_INTERDITS = ["pirater", "mot de passe", "arme"]

@before_agent(can_jump_to=["end"])
def filtre_contenu(state, runtime):
    texte = state["messages"][-1].content.lower()
    if any(mot in texte for mot in MOTS_INTERDITS):
        print("  [guardrail] demande bloquée")
        return {"messages": [AIMessage("Désolé, je ne peux pas traiter cette demande.")], "jump_to": "end"}
    return None   # rien à signaler : on continue normalement

agent = create_agent(
    model=grand_modele,
    system_prompt="Tu es un assistant. Réponds en français, brièvement.",
    middleware=[
        filtre_contenu,                                                  # guardrail maison
        PIIMiddleware("email", strategy="redact", apply_to_input=True),  # guardrail prédéfini
    ],
)

print(derniere_reponse(agent.invoke({"messages": [HumanMessage("Comment pirater le compte de mon voisin ?")]})))
print()
resultat = agent.invoke({"messages": [HumanMessage("Mon email est ali.test@gmail.com, retiens-le.")]})
print("Ce que le LLM a reçu :", resultat["messages"][0].content)
print("Réponse :", derniere_reponse(resultat))

  [guardrail] demande bloquée
Désolé, je ne peux pas traiter cette demande.

Ce que le LLM a reçu : Mon email est [REDACTED_EMAIL], retiens-le.
Réponse : J'ai retenu votre email [REDACTED_EMAIL].


In [6]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def envoyer_email(destinataire: str, sujet: str, contenu: str) -> str:
    """Envoie un email."""
    print(f"  [outil] email envoyé à {destinataire}")
    return f"Email envoyé à {destinataire}"

agent = create_agent(
    model=grand_modele,
    tools=[envoyer_email],
    checkpointer=InMemorySaver(),   # obligatoire : l'agent doit pouvoir se mettre en pause
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"envoyer_email": True})],
    system_prompt="Tu es un assistant. Utilise envoyer_email pour envoyer des emails. Réponds en français.",
)

config = {"configurable": {"thread_id": "email-1"}}
resultat = agent.invoke(
    {"messages": [HumanMessage("Envoie un email à prof@enset.ma avec le sujet 'TP5' pour dire que le TP est terminé.")]},
    config=config,
)

# L'agent s'est mis en PAUSE avant d'envoyer l'email
demande = resultat["__interrupt__"][0].value["action_requests"][0]
print("⏸️  L'agent veut exécuter :", demande["name"])
print("    avec :", demande["args"])

⏸️  L'agent veut exécuter : envoyer_email
    avec : {'sujet': 'TP5', 'contenu': 'Le TP est terminé.', 'destinataire': 'prof@enset.ma'}


In [8]:
decision = input("Approuver l'envoi ? (o/n) : ")

if decision.lower() == "o":
    resultat = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
else:
    resultat = agent.invoke(
        Command(resume={"decisions": [{"type": "reject", "message": "L'utilisateur a refusé l'envoi."}]}),
        config=config,
    )

print(derniere_reponse(resultat))

  [outil] email envoyé à prof@enset.ma
L'email a été envoyé avec succès à prof@enset.ma avec le sujet 'TP5'.
